# 🧬 04. Modelos & Arquiteturas do XFakeSong

Este notebook **descreve e implementa** cada uma das **14 arquiteturas** de
detecção de áudio *deepfake* do sistema. Para cada modelo há:

- 📖 **Descrição** — paper de referência, intuição e contribuição principal;
- 🔌 **Entrada** — áudio bruto (*raw waveform*) vs. espectrograma vs. features tabulares;
- 🧪 **Implementação** — célula que instancia o modelo via a *factory* oficial e
  resume a topologia (parâmetros, shapes, tipos de camada).

> As células são **independentes** — rode apenas os modelos que quiser inspecionar.
> Modelos grandes (HuBERT ~91M, SpectrogramTransformer ~100M, Res2Net ~49M) podem
> demorar a construir.

### Índice
| # | Modelo | Entrada | Família | Referência |
|---|--------|:-------:|---------|------------|
| 1 | AASIST | raw | Grafo espectro-temporal | Jung et al., ICASSP 2022 |
| 2 | RawNet2 | raw | SincNet + FMS + GRU | Jung et al., 2020 |
| 3 | RawGAT-ST | raw | Grafo espectral+temporal | Tak et al., 2021 |
| 4 | WavLM | raw | SSL (Transformer) | Chen et al., 2022 |
| 5 | HuBERT | raw | SSL (Transformer) | Hsu et al., 2021 |
| 6 | Sonic Sleuth | espectro | CNN + LFCC/MFCC/CQT | (98,27% acc) |
| 7 | EfficientNet-LSTM | espectro | Transfer + BiLSTM | lit. híbrida |
| 8 | MultiscaleCNN (Res2Net) | espectro | Multi-escala | Gao et al., 2019 |
| 9 | SpectrogramTransformer | espectro | ViT/AST | Gong et al., 2021 |
| 10 | Conformer | espectro | Conv + Self-Attention | Gulati et al., 2020 |
| 11 | Hybrid CNN-Transformer | espectro | CCT | Hassani et al., 2021 |
| 12 | Ensemble | espectro | Fusão multi-front-end | Pham et al., 2024 |
| 13 | SVM | tabular | ML clássico | — |
| 14 | Random Forest | tabular | ML clássico | — |


## ⚙️ 1. Configuração de Ambiente
Detecta Colab vs. local, ajusta o `PYTHONPATH` e importa a *factory* de modelos.

In [ ]:
import sys, os
from pathlib import Path

# Colab vs. local
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
    PROJECT_PATH = Path("/content/drive/MyDrive/TCC")
    print("✅ Google Drive montado.")
except ImportError:
    IN_COLAB = False
    PROJECT_PATH = Path(os.getcwd())
    # se rodar de notebooks/, sobe um nível até a raiz do projeto
    if PROJECT_PATH.name == "notebooks":
        PROJECT_PATH = PROJECT_PATH.parent
    print("⚠️ Execução local.")

if str(PROJECT_PATH) not in sys.path:
    sys.path.insert(0, str(PROJECT_PATH))
os.chdir(PROJECT_PATH)
print("Raiz do projeto:", PROJECT_PATH)

In [ ]:
# (Colab) instalar dependências — descomente se necessário
# !pip install -q -r requirements.txt

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import tensorflow as tf
from collections import Counter

from app.domain.models.architectures.factory import create_model_by_name
from app.domain.models.architectures.registry import architecture_registry as REG

# Shapes canônicos do sistema (3 s @ 16 kHz)
RAW_SHAPE  = (48000, 1)      # áudio bruto
SPEC_SHAPE = (375, 80, 1)    # espectrograma/LFCC: 375 frames × 80 bins
NUM_CLASSES = 2              # real (0) vs fake (1)
print("TensorFlow:", tf.__version__)

In [ ]:
def describe(display_name, input_shape, variant="default", num_classes=NUM_CLASSES):
    """Instancia um modelo via factory e imprime um resumo compacto da topologia."""
    info = REG.get_architecture(display_name)
    itype = info.input_requirements.get("input_type", "?")
    try:
        model = create_model_by_name(
            display_name, input_shape=input_shape,
            num_classes=num_classes, variant=variant,
        )
    except Exception as e:
        print(f"❌ {display_name} [{variant}] falhou: {type(e).__name__}: {e}")
        return None
    types = Counter(type(l).__name__ for l in model.layers)
    top = ", ".join(f"{k}×{v}" for k, v in types.most_common(7))
    print(f"✅ {display_name}  [variant={variant}]")
    print(f"   entrada={itype:11} | params={model.count_params()/1e6:.2f}M "
          f"| in={model.input_shape} → out={model.output_shape}")
    print(f"   camadas: {top}")
    return model

print("helper `describe()` pronto.")

---
## 🌊 2. Modelos *end-to-end* sobre áudio bruto
Operam diretamente na forma de onda (`raw waveform`), sem espectrograma manual.
São o estado-da-arte em anti-spoofing.

### 2.1 AASIST — *Audio Anti-Spoofing using Integrated Spectro-Temporal GAT*
> Jung et al., **ICASSP 2022**.

**Intuição.** Aprende filtros passa-banda diretamente do sinal (**SincConv**),
extrai um mapa tempo×canal com blocos residuais e o trata como **dois grafos**:
um **espectral** (canais = nós) e um **temporal** (frames = nós). A camada
**HS-GAL** (*Heterogeneous Stacking Graph Attention*) faz atenção **cruzada**
entre os dois domínios — a contribuição central do paper — seguida de
*graph pooling*, *readout* (max + atenção) e classificação **AM-Softmax**
(margem angular).

**Por que funciona p/ fake:** artefatos de vocoder aparecem como inconsistências
conjuntas em frequência **e** tempo; o grafo heterogêneo captura essa relação.

In [ ]:
aasist = describe("AASIST", RAW_SHAPE, variant="aasist")

### 2.2 RawNet2 — *Improved RawNet with Feature Map Scaling*
> Jung et al., **2020**.

**Intuição.** **SincNet** (filtros senoidais aprendíveis) → **6 blocos residuais**
com **FMS** (*Feature Map Scaling* — recalibra cada mapa por um vetor sigmoide,
estilo squeeze-excitation) → **GRU** para modelar a dinâmica temporal → FC.
O **FMS** é a contribuição-chave: enfatiza canais discriminativos para spoofing.

In [ ]:
rawnet2 = describe("RawNet2", RAW_SHAPE, variant="rawnet2")

### 2.3 RawGAT-ST — *End-to-End Spectro-Temporal Graph Attention*
> Tak et al., **2021**.

**Intuição.** Precursor do AASIST: **SincConv** sobre áudio bruto + encoder
residual, depois **dois grafos GAT** — espectral (Gs) e temporal (Gt) — fundidos
por **multiplicação element-wise** (a *graph combination* do paper). Mais enxuto
que o AASIST (sem HS-GAL).

> ⚠️ Esta é a versão **reescrita fiel ao paper** (P2): antes recebia espectrograma
> e simulava o SincNet com `Conv2D` — agora opera em áudio bruto com SincConv real.

In [ ]:
rawgat = describe("RawGAT-ST", RAW_SHAPE, variant="rawgat_st")

### 2.4 WavLM — *Self-Supervised Learning front-end*
> Chen et al., **2022**.

**Intuição.** Usa um **backbone WavLM pré-treinado** (Transformer treinado em
milhares de horas de fala) como extrator; combina os *hidden-states* de todas as
camadas por **soma ponderada aprendível** (convenção SUPERB) e alimenta um
back-end classificador. Receita SOTA atual (Tak et al. 2022): **fine-tuning
parcial** do backbone (variante padrão descongela as últimas 3 camadas).

- `variant="wavlm"` → back-end raso (conv + attention pooling);
- `variant="wavlm_aasist"` → back-end de **grafo AASIST** (receita SOTA).

> Sem o pacote `transformers`, cai num **extrator CNN 1D** equivalente (fallback).

In [ ]:
wavlm = describe("WavLM", RAW_SHAPE, variant="wavlm")
wavlm_aasist = describe("WavLM", RAW_SHAPE, variant="wavlm_aasist")

### 2.5 HuBERT — *Hidden-Unit BERT (SSL)*
> Hsu et al., **2021**.

**Intuição.** Mesma estratégia do WavLM, com o backbone **HuBERT**
(`facebook/hubert-base-ls960`). Soma ponderada de *hidden-states* + back-end
(`hubert` raso ou `hubert_aasist` em grafo). Também suporta fine-tuning parcial
e fallback CNN.

In [ ]:
hubert = describe("HuBERT", RAW_SHAPE, variant="hubert")
hubert_aasist = describe("HuBERT", RAW_SHAPE, variant="hubert_aasist")

---
## 🎛️ 3. Modelos baseados em espectrograma
Recebem `(375, 80, 1)`. **Front-end padrão = LFCC** (melhoria P0; a literatura
ASVspoof mostra LFCC > mel para spoofing, pois preserva resolução em altas
frequências onde ficam os artefatos de vocoder). Treino usa **SpecAugment** (P1).

### 3.1 Sonic Sleuth
**Intuição.** CNN compacta com **extração interna de LFCC/MFCC/CQT** (camadas
próprias) seguida de blocos convolucionais + densas. Front-end correto para
spoofing embutido no modelo (98,27% acc / 0,016 EER no conjunto do paper).
Variantes: `sonic_sleuth_mfcc`, `sonic_sleuth_cqt`, `sonic_sleuth_lfcc_cqt`.

In [ ]:
sonic = describe("Sonic Sleuth", SPEC_SHAPE, variant="sonic_sleuth")

### 3.2 EfficientNet-LSTM
**Intuição.** *Transfer learning*: backbone **EfficientNet-B0** (pré-treinado em
ImageNet) extrai features 2D do espectrograma; uma **BiLSTM [256,128]** modela a
dependência temporal. Bom equilíbrio acurácia/custo.

In [ ]:
eff = describe("EfficientNet-LSTM", SPEC_SHAPE, variant="efficientnet_lstm")

### 3.3 MultiscaleCNN — Res2Net
> Gao et al., **TPAMI 2021**.

**Intuição.** **Res2Net** introduz conexões residuais **hierárquicas dentro do
bloco** (divide os canais em sub-grupos processados em cascata), criando campos
receptivos multi-escala. Config Res2Net-50 (`scale=8, baseWidth=26, [3,4,6,3]`).
Pesado (~49M params) — há a variante `multiscale_cnn_lite`.

In [ ]:
res2net = describe("MultiscaleCNN", SPEC_SHAPE, variant="multiscale_cnn")

### 3.4 SpectrogramTransformer (ViT / AST)
> Gong et al. (**AST**), 2021.

**Intuição.** Trata o espectrograma como imagem: um **ConvStem** + **patch
embedding** (patches 8×8) gera tokens, somados a um *class token* e processados
por blocos **Transformer** (self-attention global). Captura dependências de longo
alcance no plano tempo-frequência.

In [ ]:
spectran = describe("SpectrogramTransformer", SPEC_SHAPE, variant="spectrogram_transformer")

### 3.5 Conformer
> Gulati et al., **2020**.

**Intuição.** Combina **convolução** (contexto local) com **self-attention**
(contexto global). Bloco *macaron*: `½·FF → MHSA(rel-pos) → Conv depthwise →
½·FF → LayerNorm`, com **convolutional subsampling** (4×) na entrada. Forte para
padrões locais (transições de fonema) e globais (prosódia).

In [ ]:
conformer = describe("Conformer", SPEC_SHAPE, variant="conformer")

### 3.6 Hybrid CNN-Transformer (CCT)
> Hassani et al. (**Compact Convolutional Transformer**), 2021.

**Intuição.** Um **tokenizador convolucional** substitui o *patchify* rígido do
ViT, dando viés indutivo local + eficiência de dados; em seguida um Transformer
com *sequence pooling*. Bom quando há poucos dados de treino.

In [ ]:
hybrid = describe("Hybrid CNN-Transformer", SPEC_SHAPE, variant="hybrid_cnn_transformer")

### 3.7 Ensemble multi-front-end
> Pham et al., 2024.

**Intuição.** **Quatro ramos** processam representações complementares
(**Mel + LFCC + CQT + MFCC**) e suas saídas são **fundidas** (média ponderada /
*soft-voting* / atenção). Robustez por diversidade de front-ends. Variantes:
`ensemble_score`, `ensemble_adaptive`, `ensemble_lite`. (Usa STFT compartilhado
para acelerar ~3-4×.)

In [ ]:
ensemble = describe("Ensemble", SPEC_SHAPE, variant="ensemble")

---
## 📊 4. Modelos clássicos (features tabulares)
Não usam espectrograma/áudio bruto diretamente: consomem um **vetor de features**
agregadas (espectrais + cepstrais + temporais) extraídas por segmento e
concatenadas. `StandardScaler` + classificador scikit-learn.

### 4.1 SVM
**Intuição.** `StandardScaler` → **SVC** com kernel **RBF** (`probability=True`
para gerar probabilidades calibradas). Forte baseline com poucas amostras;
fronteira de decisão não-linear no espaço de features.

In [ ]:
from app.domain.models.architectures import svm as svm_mod

# dim tabular típica do pipeline (spectral+cepstral+temporal por segmento)
TAB_SHAPE = (7616,)
svm_model = svm_mod.create_model(input_shape=TAB_SHAPE, num_classes=NUM_CLASSES)
print("SVM:", type(svm_model).__name__, "| pipeline:", svm_model.pipeline.named_steps
      if hasattr(svm_model, "pipeline") else "(StandardScaler + SVC rbf)")

### 4.2 Random Forest
**Intuição.** `StandardScaler` → **RandomForestClassifier** (ensemble de árvores,
`n_estimators=100`, `n_jobs=-1`). Robusto a *outliers*, fornece **importância de
features** (útil para XAI). Bom baseline interpretável.

In [ ]:
from app.domain.models.architectures.random_forest import create_random_forest_model

rf_model = create_random_forest_model(input_shape=TAB_SHAPE, num_classes=NUM_CLASSES,
                                      n_estimators=100)
print("Random Forest:", type(rf_model).__name__, "| n_estimators=100, n_jobs=-1")

---
## 🧾 5. Resumo comparativo (gerado do *registry*)
Lê o *registry* oficial e lista todas as arquiteturas DL com tipo de entrada e
variantes registradas — fonte da verdade do sistema.

In [ ]:
rows = []
for name, info in REG._architectures.items():
    rows.append((name,
                 info.input_requirements.get("input_type", "?"),
                 ", ".join(info.supported_variants[:4])))
print(f"{'Modelo':24} {'Entrada':12} Variantes")
print("-" * 80)
for n, it, v in rows:
    print(f"{n:24} {it:12} {v}")
print(f"\nTotal DL no registry: {len(rows)}  (+ SVM e Random Forest = 14 modelos)")

---
## 🚀 6. Como treinar qualquer um destes modelos

Três caminhos equivalentes (todos gravam o **`input_contract`** que garante que a
inferência reproduza exatamente o pré-processamento do treino):

1. **UI Gradio** → aba **🎓 Treinar → 🪄 Assistente** (wizard guiado em 4 passos).
2. **`TrainingService`** (programático) — ver `notebooks/02_Training.ipynb`.
3. **Factory direta** (este notebook) — `create_model_by_name(...)` + `model.fit(...)`.

### Melhorias de pré-processamento já integradas (revisão vs. literatura)
| Melhoria | Onde | Modelos beneficiados |
|----------|------|----------------------|
| **RawBoost** (Tak 2022) | augmentation de áudio bruto (treino) | AASIST, RawNet2, RawGAT-ST, WavLM, HuBERT |
| **LFCC** front-end padrão | `feature_frontend="lfcc"` no contrato | todos os de espectrograma |
| **SpecAugment** (Park 2019) | mascaramento tempo/freq (treino) | todos os de espectrograma |
| **SSL fine-tuning parcial** | `n_trainable_layers` | WavLM, HuBERT |
| **SSL → back-end AASIST** | variantes `*_aasist` | WavLM, HuBERT |

> Detalhes da conformidade com os papers em **`docs/14_REVISAO_ARQUITETURAS.md`**.

In [ ]:
# Exemplo mínimo de treino com dados sintéticos (substitua por dados reais)
import numpy as np
X = np.random.randn(16, *RAW_SHAPE).astype("float32")   # 16 clipes
y = np.random.randint(0, 2, size=16)

model = create_model_by_name("AASIST", input_shape=RAW_SHAPE, num_classes=2, variant="aasist")
model.fit(X, y, epochs=1, batch_size=4, verbose=1)
print("✅ 1 época de exemplo concluída (use seu dataset real para treinar de fato).")

---
✦ **Notebook gerado para documentação técnica do XFakeSong.**
Próximos passos: `02_Training.ipynb` (treino completo) · `03_Inference_and_Demo.ipynb` (inferência).